In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless pillow matplotlib gdown pyyaml

In [ ]:
import glob
import os
import shutil
import yaml
import gdown

# 1. Setup paths
extract_dir = "/content/YOLO_Crab_Project"
zip_path = "/content/All-Data-1.zip"

# Clean up existing files/directories
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
if os.path.exists(zip_path):
    os.remove(zip_path)

os.makedirs(extract_dir, exist_ok=True)

# 2. Set file ID and download
file_id = "1KCT_jYauMnJKNz20MRc09hyHBb8ghRek"
download_url = f"https://drive.google.com/uc?id={file_id}"

print("Downloading All-Data-1.zip...")
gdown.download(download_url, zip_path, quiet=False)

# 3. Extract the ZIP file
print("Extracting files...")
shutil.unpack_archive(zip_path, extract_dir)
os.remove(zip_path)

# 4. Locate and load data.yaml
yaml_files = glob.glob(f"{extract_dir}/**/data.yaml", recursive=True)

if not yaml_files:
    raise FileNotFoundError(
        f"Could not find data.yaml inside {extract_dir}."
    )

data_yaml_path = yaml_files[0]
dataset_root = os.path.dirname(data_yaml_path)

print("\n--- Success ---")
print("Found data.yaml at:", data_yaml_path)
print("Dataset root:", dataset_root)

with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("\nDataset configuration:")
print(data_config)

In [ ]:
import glob
import os
import yaml

extract_dir = "/content/YOLO_Crab_Project"

# Locate data.yaml inside the extracted dataset directory
yaml_files = glob.glob(f"{extract_dir}/**/data.yaml", recursive=True)

if not yaml_files:
    raise FileNotFoundError(
        f"Could not find data.yaml inside {extract_dir}."
    )

data_yaml_path = yaml_files[0]
dataset_root = os.path.dirname(data_yaml_path)

# Update dataset root path in data.yaml
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

data_config["path"] = dataset_root  # Absolute path to extracted dataset root

with open(data_yaml_path, "w") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"Found data.yaml at: {data_yaml_path}")
print(f"Dataset root set to: {dataset_root}")

In [ ]:
from ultralytics import YOLO

# Load a valid pretrained YOLO model (e.g., yolo11s.pt or yolov8s.pt)
model = YOLO("yolo11s.pt")

# Train model
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    workers=2,
    patience=20,
)

print("Training completed! Check runs/detect/train/weights/best.pt for outputs.")

In [ ]:
import os
from IPython.display import Image, display
from ultralytics import YOLO

# 1. Load trained model
model_path = "runs/detect/train/weights/best.pt"
model = YOLO(model_path)

# 2. Run prediction
image_source = "crab.jpg"
results = model.predict(source=image_source, conf=0.40, save=True)

# 3. Dynamically resolve output path
save_dir = results[0].save_dir  # Returns actual directory (e.g., runs/detect/predict2)
image_name = os.path.basename(image_source)  # Keeps original filename ("crab.png")
output_path = os.path.join(save_dir, image_name)

# 4. Display annotated result
if os.path.exists(output_path):
    display(Image(filename=output_path))
else:
    print(f"Could not find output image at: {output_path}")

In [ ]:
from ultralytics import YOLO

model_path = "runs/detect/train/weights/best.pt"
model = YOLO(model_path)

metrics = model.val(data=str(data_yaml_path), split="val")

print("\n--- Accuracy Summary ---")
print(f"Precision:    {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall:       {metrics.results_dict['metrics/recall(B)']:.4f}")